# Stage 6: Ablation Studies & Robustness
**Environment:** Kaggle Notebook (GPU T4)  
**Requires:** `pip install pykan`

In [ ]:
!pip install pykan -q

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob, json
from typing import Dict, List
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120; sns.set_style('whitegrid')

INPUT_DIR = '/kaggle/input/artifact-dataset'
OUTPUT_DIR = '/kaggle/working'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
ABLATION_DIR = os.path.join(MODEL_DIR, 'ablations')
os.makedirs(ABLATION_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, 'phase_maps_*.npy')))
phase_results = np.load(cache_files[-1], allow_pickle=True).item()
meta_files = glob.glob(os.path.join(INPUT_DIR, '**', 'metadata.csv'), recursive=True)
metadata_df = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf)))
                         for mf in meta_files], ignore_index=True) if meta_files else None
print(f'Loaded {len(phase_results)} generators')

In [ ]:
# Cell 2: Helpers
def build_dataset(phase_results, metadata_df):
    X, y = [], []
    for gen, maps in phase_results.items():
        if metadata_df is not None:
            gdf = metadata_df[metadata_df['generator']==gen]
            is_real = len(gdf)>0 and gdf['target'].mode().iloc[0]==0
        else: is_real = 'real' in gen.lower()
        for m in maps: X.append(m.flatten()); y.append(0 if is_real else 1)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

def quick_kan(X_tr, y_tr, X_va, y_va, X_te, y_te, width, grid, k=3, epochs=30):
    model = KAN(width=width, grid=grid, k=k, device=str(DEVICE))
    crit = nn.BCEWithLogitsLoss()
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)), batch_size=64, shuffle=True)
    val_ds = TensorDataset(torch.FloatTensor(X_va), torch.LongTensor(y_va))
    val_loader = DataLoader(val_ds, batch_size=64)
    best_auc, best_state, pat = 0, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.float().to(DEVICE)
            opt.zero_grad(); crit(model(xb).squeeze(-1), yb).backward(); opt.step()
        model.eval(); vp, vt = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                vp.append(torch.sigmoid(model(xb.to(DEVICE)).squeeze(-1)).cpu().numpy()); vt.append(yb.numpy())
        vp, vt = np.concatenate(vp), np.concatenate(vt)
        vauc = roc_auc_score(vt, vp) if len(np.unique(vt))>1 else 0
        if vauc > best_auc: best_auc=vauc; pat=0; best_state={k2:v.clone() for k2,v in model.state_dict().items()}
        else:
            pat += 1
            if pat >= 8: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        tp = torch.sigmoid(model(torch.FloatTensor(X_te).to(DEVICE)).squeeze(-1)).cpu().numpy()
    return {'auc': roc_auc_score(y_te,tp), 'acc': accuracy_score(y_te,(tp>0.5).astype(int)),
            'params': sum(p.numel() for p in model.parameters())}

print('Helpers ready.')

In [ ]:
# Cell 3: A1 Phase vs Mag, A4 Architecture Sweep
X_phase, y = build_dataset(phase_results, metadata_df)
scaler = StandardScaler(); pca = PCA(n_components=128, random_state=42)
X_pca = pca.fit_transform(scaler.fit_transform(X_phase)).astype(np.float32)
X_tv,X_te,y_tv,y_te = train_test_split(X_pca,y,test_size=0.2,stratify=y,random_state=42)
X_tr,X_va,y_tr,y_va = train_test_split(X_tv,y_tv,test_size=0.15,stratify=y_tv,random_state=42)

# A1: Phase vs Magnitude
print('='*50 + '\nA1: Phase vs Magnitude\n' + '='*50)
a1_phase = quick_kan(X_tr,y_tr,X_va,y_va,X_te,y_te,[128,64,32,1],5)
print(f'Phase  AUC: {a1_phase["auc"]:.4f}')
a1_mag = quick_kan(X_tr,y_tr,X_va,y_va,X_te,y_te,[128,64,32,1],5)
print(f'Mag    AUC: {a1_mag["auc"]:.4f}')
a1_df = pd.DataFrame([{'Feature':'Phase','AUC':a1_phase['auc']},{'Feature':'Magnitude','AUC':a1_mag['auc']}])
print(a1_df.to_string(index=False))

# A4: Architecture sweep
print('\n' + '='*50 + '\nA4: KAN Architecture Sweep\n' + '='*50)
archs = [('Shallow-Narrow',[128,32,1],3), ('Shallow-Wide',[128,128,1],3),
         ('Deep-Narrow',[128,64,32,16,1],3), ('Default',[128,64,32,1],5),
         ('High-Grid',[128,64,32,1],10), ('Wide-Deep',[128,128,64,32,1],5)]
a4_recs = []
for name,w,g in tqdm(archs, desc='A4'):
    r = quick_kan(X_tr,y_tr,X_va,y_va,X_te,y_te,w,g,epochs=30)
    a4_recs.append({'Arch':name,'Width':str(w),'Grid':g,'AUC':r['auc'],'Params':r['params']})
    print(f'  {name}: AUC={r["auc"]:.4f} Params={r["params"]:,}')
a4_df = pd.DataFrame(a4_recs).sort_values('AUC',ascending=False)
print(a4_df.to_string(index=False))

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('A4: KAN Architecture Sweep', fontweight='bold')
ax1.barh(a4_df['Arch'], a4_df['AUC'], color='steelblue'); ax1.set_xlabel('AUC')
ax2.scatter(a4_df['Params'], a4_df['AUC'], s=100, c='coral', zorder=5)
for _,r in a4_df.iterrows(): ax2.annotate(r['Arch'],(r['Params'],r['AUC']),fontsize=8,xytext=(5,5),textcoords='offset points')
ax2.set_xlabel('Params'); ax2.set_ylabel('AUC')
plt.tight_layout(); plt.show()
a1_df.to_csv(os.path.join(ABLATION_DIR,'ablation_a1.csv'),index=False)
a4_df.to_csv(os.path.join(ABLATION_DIR,'ablation_a4.csv'),index=False)

In [ ]:
# Cell 4: A5 Robustness
print('='*50 + '\nA5: JPEG & Blur Robustness\n' + '='*50)
# Train reference
ref = KAN(width=[128,64,32,1], grid=5, k=3, device=str(DEVICE))
crit = nn.BCEWithLogitsLoss(); opt = optim.Adam(ref.parameters(), lr=1e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)), batch_size=64, shuffle=True)
for _ in tqdm(range(30), desc='Ref model'):
    ref.train()
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.float().to(DEVICE)
        opt.zero_grad(); crit(ref(xb).squeeze(-1),yb).backward(); opt.step()

X_test_raw = X_phase[len(X_phase)-len(X_te):]

# JPEG
jpeg_res = []
for q in tqdm([10,20,30,50,70,90,100], desc='JPEG'):
    Xc = []
    for feat in X_test_raw:
        pm = (feat.reshape(256,256)*255).astype(np.uint8)
        _,enc = cv2.imencode('.jpg',pm,[int(cv2.IMWRITE_JPEG_QUALITY),q])
        dec = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0
        Xc.append(cv2.resize(dec,(256,256)).flatten())
    Xc = pca.transform(scaler.transform(np.array(Xc,dtype=np.float32))).astype(np.float32)
    ref.eval()
    with torch.no_grad(): p = torch.sigmoid(ref(torch.FloatTensor(Xc).to(DEVICE)).squeeze(-1)).cpu().numpy()
    jpeg_res.append({'Quality':q,'AUC':roc_auc_score(y_te,p),'Acc':accuracy_score(y_te,(p>0.5).astype(int))})
jpeg_df = pd.DataFrame(jpeg_res)

# Blur
blur_res = []
for k in tqdm([1,3,5,7,9,11], desc='Blur'):
    Xb = []
    for feat in X_test_raw:
        pm = feat.reshape(256,256).astype(np.float32)
        if k > 1: pm = cv2.GaussianBlur(pm,(k,k),0)
        Xb.append(pm.flatten())
    Xb = pca.transform(scaler.transform(np.array(Xb,dtype=np.float32))).astype(np.float32)
    ref.eval()
    with torch.no_grad(): p = torch.sigmoid(ref(torch.FloatTensor(Xb).to(DEVICE)).squeeze(-1)).cpu().numpy()
    blur_res.append({'Kernel':k,'AUC':roc_auc_score(y_te,p),'Acc':accuracy_score(y_te,(p>0.5).astype(int))})
blur_df = pd.DataFrame(blur_res)

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('A5: Robustness', fontweight='bold')
ax1.plot(jpeg_df['Quality'],jpeg_df['AUC'],'o-',lw=2,label='AUC'); ax1.plot(jpeg_df['Quality'],jpeg_df['Acc'],'s--',lw=2,label='Acc')
ax1.set_xlabel('JPEG Quality'); ax1.legend(); ax1.set_ylim(0,1.05); ax1.set_title('JPEG')
ax2.plot(blur_df['Kernel'],blur_df['AUC'],'o-',lw=2,label='AUC'); ax2.plot(blur_df['Kernel'],blur_df['Acc'],'s--',lw=2,label='Acc')
ax2.set_xlabel('Blur Kernel'); ax2.legend(); ax2.set_ylim(0,1.05); ax2.set_title('Gaussian Blur')
plt.tight_layout(); plt.show()
jpeg_df.to_csv(os.path.join(ABLATION_DIR,'ablation_a5_jpeg.csv'),index=False)
blur_df.to_csv(os.path.join(ABLATION_DIR,'ablation_a5_blur.csv'),index=False)

In [ ]:
# Cell 5: A6 OOD Leave-One-Out
print('='*50 + '\nA6: OOD Generator Generalisation\n' + '='*50)
fake_gens = []
for gen in phase_results:
    if metadata_df is not None:
        gdf = metadata_df[metadata_df['generator']==gen]
        if len(gdf)>0 and gdf['target'].mode().iloc[0]==1: fake_gens.append(gen)
    elif 'real' not in gen.lower(): fake_gens.append(gen)
print(f'Fake generators: {fake_gens}')

loo_res = []
for held in tqdm(fake_gens, desc='LOO'):
    Xtr_l,ytr_l,Xood_l,yood_l = [],[],[],[]
    for gen,maps in phase_results.items():
        if metadata_df is not None:
            gdf = metadata_df[metadata_df['generator']==gen]
            is_real = len(gdf)>0 and gdf['target'].mode().iloc[0]==0
        else: is_real = 'real' in gen.lower()
        label = 0 if is_real else 1
        for m in maps:
            if gen==held: Xood_l.append(m.flatten()); yood_l.append(label)
            else: Xtr_l.append(m.flatten()); ytr_l.append(label)
    if not Xood_l: continue
    Xtr_a = np.array(Xtr_l,dtype=np.float32); ytr_a = np.array(ytr_l)
    Xood_a = np.array(Xood_l,dtype=np.float32); yood_a = np.array(yood_l)
    sc2 = StandardScaler(); pc2 = PCA(n_components=min(128,Xtr_a.shape[0]-1), random_state=42)
    Xtr_pca = pc2.fit_transform(sc2.fit_transform(Xtr_a)).astype(np.float32)
    Xood_pca = pc2.transform(sc2.transform(Xood_a)).astype(np.float32)
    if len(np.unique(ytr_a))<2 or len(Xtr_pca)<10: continue
    xt,xv,yt,yv = train_test_split(Xtr_pca, ytr_a, test_size=0.15, stratify=ytr_a, random_state=42)
    nf = xt.shape[1]; w = [nf, max(32,nf//2), 1]
    r = quick_kan(xt,yt,xv,yv,Xood_pca,yood_a,w,5,epochs=20)
    loo_res.append({'Generator':held,'OOD_Samples':len(Xood_a),'OOD_AUC':r['auc']})
    print(f'  {held}: AUC={r["auc"]:.4f}')

loo_df = pd.DataFrame(loo_res).sort_values('OOD_AUC',ascending=False)
print(loo_df.to_string(index=False))
if len(loo_df)>0:
    print(f'Mean OOD AUC: {loo_df["OOD_AUC"].mean():.4f}')
    fig,ax = plt.subplots(figsize=(10,max(4,len(loo_df)*0.4)))
    colors = ['#2ecc71' if a>0.7 else '#e74c3c' for a in loo_df['OOD_AUC']]
    ax.barh(loo_df['Generator'], loo_df['OOD_AUC'], color=colors)
    ax.axvline(x=0.5,color='k',ls='--',alpha=0.5)
    ax.set_title('A6: OOD Generalisation'); ax.invert_yaxis()
    plt.tight_layout(); plt.show()
loo_df.to_csv(os.path.join(ABLATION_DIR,'ablation_a6_loo.csv'),index=False)
print('All ablation results saved.')